## Introduction ##

---
tags: 
- Numerical methods
- Quantum Mehcanics 
---


One of the computational tools that we used for modeling our hydrogen atom was the scipy.linalg library, more specifically we used the scipy.linalg.eigh() function which allows us to solve for eigenvalues and eigenvector for real, symmetric Hermitian matrices. 


### How we implement it ###

In our project we took Schrodinger's equation and after seperating into 3 seperate ODE's we are able to discretze these differential equations and apply the finite difference method to each equation we can form 3 different real symmetric matrices which turn out to be hermitian. 

Since we noticed that these matrices were real, symmetric and herimitian we were able to apply our eigh() function to allow us to extra the eigenvectors and eigenvalues of our matrix. 

## Example ## 

The basic syntax of our scipy.linalg.eigh() function is give as such 

eigenvalues, eigenvector = scipy.linalg.eigh(A) where A is our given A is our real symmetric matrix. 

For example here is a more trivial example of this function

In [2]:
import numpy as np
import scipy.linalg

A = np.array([[2, 1], [1, 2]])

eigenvalues, eigenvectors = scipy.linalg.eigh(A)

print(eigenvalues, eigenvectors)

[1. 3.] [[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


So we see this function printed our eigenvalues and eigenvectors but also to note that these eigenvectors are orthonormal allowing us to easily apply this to our matrix from finite difference method which we require orthonormal eigenvectors. 

## Example for our Hydrogen atom ##

First when constructing the our wave function we split it up into 3 parts, one for each of the component of our wavefunction. First we define a phi_operator which is a matrix formed from the finite difference method. We can then do the using our eigh() function to solve for the eigenvalues and eigenvector of our functions. Since we only care about the lowest energy solutions we then sort and truncate our eigenvalues using phi_indicies = np.argsort(phi_eigenvalues) B = phi_eigenvalues[phi_indicies][:PHI_CUTTOFF]. 

Now doing the same result with finite difference method of theta direction we can then define theta_operator which is constructed as theta_operators = [ThetaOperator(b) for b in B]. For our eigenvalues of theta_operator produces a generalized eigenvalue problem so we can account for this by theta_eig = [scipy.linalg.eigh(op[0], b=op[1]) for op in theta_operators] where op[0] is the matrix representation of the theta_operator and op[1] is the corresponding weight matrix (W). These eigenvalues like the phi_operator were collected and sorted by 
unsorted_AB = np.array([ [[b,a] for a in theta_eig[i][0]] for i,b in enumerate(B) ]).reshape((-1,2)).transpose()
theta_indicies = np.lexsort(np.round(unsorted_AB,2)) 
A = unsorted_AB[1][theta_indicies][:SPHERICAL_HARMONIC_CUTOFF]. 

Again we will do the same results but for the radial direction, we can define r_operator = [ROperator(a) for a in A] this is also another generalized eigenvalue problem so we use r_eig = [scipy.linalg.eigh(op[0], b=op[1]) for op in r_operators] so r_eig[i][0]: the eigenvalues, and r_eig[i][1]: the corresponding eigenvectors. These eigenvalues which correspond to the eigen of the hydrogen atom were converted to eV and organized using unsorted_EA = np.array([ [[B_sorted_for_A[i], a, 1/ECHARGE*e] for e in r_eig[i][0]] for i,a in enumerate(A) ]).reshape((-1,3)).transpose(), and were sorted by r_indicies = np.lexsort(np.round(unsorted_EA,2)) E = unsorted_EA[2][r_indicies][:HYDROGEN_ORBITAL_CUTOFF]. 


The full code is as shown below. 

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg

N = 500
RADIUS_FOCUS = 5.292e-11

PHI_CUTTOFF = 25
SPHERICAL_HARMONIC_CUTOFF = 64
HYDROGEN_ORBITAL_CUTOFF = 100

HBAR = 1.0546e-34
K = 1/(4*np.pi * 8.8542e-12)
MU = 9.1094e-31
ECHARGE = 1.6022e-19

def PhiOperator():
  dphi = 2*np.pi / (N)
  dif = np.diag(np.ones(N)) - np.diag(np.ones(N-1), k=-1)
  corner_one = np.zeros((N,N)); corner_one[0][N-1] = 1
  periodic_dif = dif - corner_one
  phi_operator = (periodic_dif.T + periodic_dif) * (1/dphi**2)
  return phi_operator

def ThetaOperator(B):
  dtheta = np.pi / (N+1)
  main_diagonal = np.diag([np.sin((3/2) * dtheta), *[
      (np.sin((n+1/2) * dtheta) + np.sin((n-1/2) * dtheta))
    for n in range(2,N)], np.sin(((N) - 1/2) * dtheta)])
  top_off_diagonal = np.diag([
      np.sin((n+1/2) * dtheta)
    for n in range(1,N)], k=1)
  bottom_off_diagonal = np.diag([
      np.sin((n-1/2) * dtheta)
    for n in range(2,N+1)], k=-1)
  differential_operator = (main_diagonal - bottom_off_diagonal - top_off_diagonal) * (1/dtheta**2)

  B_term = np.diag([
      B / np.sin((n) * dtheta)
    for n in range(1,N+1)])
  theta_operator = differential_operator + B_term

  W = np.diag([
      np.sin((n) * dtheta)
    for n in range(1,N+1)])
  return (theta_operator, W)

def ROperator(A, Z=1):
  r0 = RADIUS_FOCUS
  drho = (np.pi/2) / (N+1)
  main_diagonal = np.diag([r0 * np.sin((3/2) * drho)**2, *[
      (r0 * np.sin((n+1/2) * drho)**2 + r0 * np.sin((n-1/2) * drho)**2)
    for n in range(2,N)], r0 * np.sin(((N) - 1/2) * drho)**2])
  top_off_diagonal = np.diag([
      r0 * np.sin((n+1/2) * drho)**2
    for n in range(1,N)], k=1)
  bottom_off_diagonal = np.diag([
      r0 * np.sin((n-1/2) * drho)**2
    for n in range(2,N+1)], k=-1)
  differential_operator = (main_diagonal - bottom_off_diagonal - top_off_diagonal) * (1/drho**2)

  potential_term = np.diag([
      -(2*MU*K*Z*ECHARGE**2/HBAR**2) * r0**2 * np.sin((n) * drho) / np.cos((n) * drho)**3
    for n in range(1,N+1)])
  A_term = np.diag([
      A * r0 / np.cos((n) * drho)**2
    for n in range(1,N+1)])
  r_operator = differential_operator + potential_term + A_term

  W = np.diag([
      (2 * MU / HBAR**2) * r0**3 * np.sin((n) * drho)**2 / np.cos((n) * drho)**4
    for n in range(1,N+1)])
  return (r_operator, W)


# Solve the phi equation
phi_operator = PhiOperator()
phi_eigenvalues, phi_eigenstates = scipy.linalg.eigh(phi_operator)
phi_eigenstates = phi_eigenstates.transpose()
phi_indicies = np.argsort(phi_eigenvalues)
B = phi_eigenvalues[phi_indicies][:PHI_CUTTOFF]

# Solve the theta equation
theta_operators = [ThetaOperator(b) for b in B]
theta_eig = [scipy.linalg.eigh(op[0], b=op[1]) for op in theta_operators]
theta_eigenstates = np.array([eig[1].transpose() for eig in theta_eig]).reshape((-1,N))
unsorted_AB = np.array([[[b,a] for a in theta_eig[i][0]] for i,b in enumerate(B)]).reshape((-1,2)).transpose()
theta_indicies = np.lexsort(np.round(unsorted_AB, 2))
A = unsorted_AB[1][theta_indicies][:SPHERICAL_HARMONIC_CUTOFF]
phi_indicies_for_A = phi_indicies[theta_indicies // N]
B_sorted_for_A = phi_eigenvalues[phi_indicies_for_A][:SPHERICAL_HARMONIC_CUTOFF]

# Solve the r equation
r_operators = [ROperator(a) for a in A]
r_eig = [scipy.linalg.eigh(op[0], b=op[1]) for op in r_operators]
r_eigenstates = np.array([eig[1].transpose() for eig in r_eig]).reshape((-1,N))
unsorted_EA = np.array([[[B_sorted_for_A[i], a, 1/ECHARGE*e] for e in r_eig[i][0]] for i,a in enumerate(A)]).reshape((-1,3)).transpose()
r_indicies = np.lexsort(np.round(unsorted_EA, 2))

# Gather the eigenvalues
E = unsorted_EA[2][r_indicies][:HYDROGEN_ORBITAL_CUTOFF]
theta_indicies_for_E = theta_indicies[r_indicies // N]
A_sorted_for_E = unsorted_AB[1][theta_indicies_for_E][:HYDROGEN_ORBITAL_CUTOFF]
phi_indicies_for_E = phi_indicies_for_A[r_indicies // N]
B_sorted_for_E = phi_eigenvalues[phi_indicies_for_E][:HYDROGEN_ORBITAL_CUTOFF]


## Conclusion ##

The main purpose of the scipy.linalg.eigh() was to solve for the eigenvalues of each component of schrodingers wave equation. Each eigenvalue tells us the state of our hydrogem atom for example phi eigenvalues represents z angular momentum, theta eigenvalues correspond to total angular momentum squared and radial eigenvalues represent energy of our hydrogen atom. 

We found these eigenvalues by turning each component's ODE into a finite difference method which we constructed matrices with that were real, symmetric and herimitian. So instead of solving the ODE's we just approximate these ODE"s to be matrices and sort the eigenvalues for the lowest possible. 